# Sun Pharma EMA / SMA Crossover

Daily Sun Pharma candlesticks with **EMA 21** and **SMA 50**. A bullish crossover is marked when EMA 21 moves above SMA 50; a bearish crossover is marked when it moves below SMA 50.

In [27]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root whether Jupyter starts there or in a notebook folder."""
    start = Path.cwd() if start is None else Path(start)
    for path in (start, *start.parents):
        if (path / "requirements.txt").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not locate the visualizer project root.")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from visualizer import load_candle_json

pd.options.display.float_format = "{:,.2f}".format

## Load Sun Pharma data

In [28]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "sunpharma_2024-08-01_2026-07-24.json"

price = load_candle_json(DATA_PATH)
required_columns = {"timestamp", "open", "high", "low", "close", "volume"}
missing_columns = required_columns.difference(price.columns)
if missing_columns:
    raise ValueError(f"Missing required column(s): {sorted(missing_columns)}")

numeric_columns = ["open", "high", "low", "close", "volume"]
price[numeric_columns] = price[numeric_columns].apply(pd.to_numeric, errors="coerce")
price = (
    price.dropna(subset=["timestamp", "open", "high", "low", "close"])
    .sort_values("timestamp")
    .drop_duplicates("timestamp", keep="last")
    .set_index("timestamp")
)

print(f"Loaded {DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"{len(price):,} daily candles from {price.index.min().date()} to {price.index.max().date()}")
display(price.head())

Loaded data/raw/sunpharma_2024-08-01_2026-07-24.json
491 daily candles from 2024-08-01 to 2026-07-24


,open,high,low,close,volume,open_interest
timestamp,,,,,,
2024-08-01 00:00:00+05:30,"1,725.50","1,746.45","1,681.30","1,715.20",5258291,0
2024-08-02 00:00:00+05:30,"1,700.00","1,741.80","1,683.85","1,731.65",2992406,0
2024-08-05 00:00:00+05:30,"1,725.00","1,758.00","1,715.00","1,720.35",5165558,0
2024-08-06 00:00:00+05:30,"1,725.00","1,734.00","1,703.95","1,707.55",1710278,0
2024-08-07 00:00:00+05:30,"1,720.00","1,736.85","1,709.40","1,734.45",1922313,0


## Calculate EMA 21, SMA 50, and crossover events

In [29]:
EMA_PERIOD = 21
SMA_PERIOD = 50
ANGLE_LOOKBACK = 3


def angle(series: pd.Series, window: int = 3) -> pd.Series:
    """Return the line angle in degrees using average price-point slope per bar."""
    numeric = pd.to_numeric(series, errors="coerce").astype(float)
    slope = (numeric - numeric.shift(window)) / float(window)
    return pd.Series(
        np.degrees(np.arctan(np.clip(slope, -10, 10))),
        index=series.index,
    )

price[f"ema_{EMA_PERIOD}"] = price["close"].ewm(
    span=EMA_PERIOD,
    adjust=False,
    min_periods=EMA_PERIOD,
).mean()
price[f"sma_{SMA_PERIOD}"] = price["close"].rolling(
    window=SMA_PERIOD,
    min_periods=SMA_PERIOD,
).mean()
price[f"ema_{EMA_PERIOD}_angle"] = angle(price[f"ema_{EMA_PERIOD}"], ANGLE_LOOKBACK)
price[f"sma_{SMA_PERIOD}_angle"] = angle(price[f"sma_{SMA_PERIOD}"], ANGLE_LOOKBACK)

ema = price[f"ema_{EMA_PERIOD}"]
sma = price[f"sma_{SMA_PERIOD}"]
price["bullish_crossover"] = (ema > sma) & (ema.shift(1) <= sma.shift(1))
price["bearish_crossover"] = (ema < sma) & (ema.shift(1) >= sma.shift(1))

price["crossover"] = pd.Series(pd.NA, index=price.index, dtype="string")
price.loc[price["bullish_crossover"], "crossover"] = "Bullish"
price.loc[price["bearish_crossover"], "crossover"] = "Bearish"

latest = price.iloc[-1]
trend = "Bullish" if latest[f"ema_{EMA_PERIOD}"] > latest[f"sma_{SMA_PERIOD}"] else "Bearish"
print(
    f"Latest close: ₹{latest['close']:,.2f} | "
    f"EMA {EMA_PERIOD}: ₹{latest[f'ema_{EMA_PERIOD}']:,.2f} | "
    f"SMA {SMA_PERIOD}: ₹{latest[f'sma_{SMA_PERIOD}']:,.2f} | "
    f"Trend: {trend}\n"
    f"{ANGLE_LOOKBACK}-bar angles — "
    f"EMA {EMA_PERIOD}: {latest[f'ema_{EMA_PERIOD}_angle']:.2f}° | "
    f"SMA {SMA_PERIOD}: {latest[f'sma_{SMA_PERIOD}_angle']:.2f}°"
)

Latest close: ₹1,941.10 | EMA 21: ₹1,918.46 | SMA 50: ₹1,866.69 | Trend: Bullish
3-bar angles — EMA 21: 71.76° | SMA 50: 63.04°


## Crossover events

In [30]:
crossover_events = (
    price.loc[
        price["bullish_crossover"] | price["bearish_crossover"],
        [
            "close",
            f"ema_{EMA_PERIOD}",
            f"sma_{SMA_PERIOD}",
            f"ema_{EMA_PERIOD}_angle",
            f"sma_{SMA_PERIOD}_angle",
            "crossover",
        ],
    ]
    .rename_axis("date")
    .reset_index()
)

print(
    f"Found {price['bullish_crossover'].sum()} bullish and "
    f"{price['bearish_crossover'].sum()} bearish crossover(s)."
)
display(crossover_events)

Found 7 bullish and 7 bearish crossover(s).


,date,close,ema_21,sma_50,ema_21_angle,sma_50_angle,crossover
0,2024-11-05 00:00:00+05:30,"1,803.60","1,863.11","1,865.57",-77.59,48.92,Bearish
1,2024-12-30 00:00:00+05:30,"1,883.90","1,820.75","1,818.09",78.04,-38.71,Bullish
2,2025-01-21 00:00:00+05:30,"1,762.70","1,801.53","1,804.51",-71.09,-32.21,Bearish
3,2025-01-24 00:00:00+05:30,"1,822.20","1,805.87","1,805.50",55.32,18.33,Bullish
4,2025-01-27 00:00:00+05:30,"1,786.85","1,804.14","1,805.66",43.44,24.45,Bearish
5,2025-03-27 00:00:00+05:30,"1,731.45","1,711.53","1,711.39",76.95,-7.39,Bullish
6,2025-05-26 00:00:00+05:30,"1,676.10","1,730.86","1,736.08",-77.04,61.18,Bearish
7,2025-07-21 00:00:00+05:30,"1,692.30","1,685.49","1,685.15",49.45,-35.98,Bullish
8,2025-08-06 00:00:00+05:30,"1,595.20","1,672.35","1,677.59",-79.90,-46.31,Bearish
9,2025-10-08 00:00:00+05:30,"1,631.60","1,624.07","1,623.60",67.29,-41.80,Bullish


## Price chart

Green triangles show bullish EMA-21 crossovers above SMA-50; red triangles show bearish crossovers below it. The angle panel uses a three-bar price-point slope and displays the result in degrees.

In [31]:
# Use None for the full history, or set a number such as 250 for recent candles only.
PLOT_BARS = None
plot_df = price.copy() if PLOT_BARS is None else price.tail(PLOT_BARS).copy()

bullish = plot_df.loc[plot_df["bullish_crossover"]]
bearish = plot_df.loc[plot_df["bearish_crossover"]]
marker_padding = max(float((plot_df["high"] - plot_df["low"]).median()) * 0.45, 1.0)
volume_colors = plot_df["close"].ge(plot_df["open"]).map(
    {True: "rgba(22, 163, 74, 0.42)", False: "rgba(220, 38, 38, 0.42)"}
)

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.68, 0.18, 0.14],
    vertical_spacing=0.04,
)

fig.add_trace(
    go.Candlestick(
        x=plot_df.index,
        open=plot_df["open"],
        high=plot_df["high"],
        low=plot_df["low"],
        close=plot_df["close"],
        name="SUNPHARMA",
        increasing_line_color="#16a34a",
        decreasing_line_color="#dc2626",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=plot_df.index,
        y=plot_df[f"ema_{EMA_PERIOD}"],
        mode="lines",
        name=f"EMA {EMA_PERIOD}",
        line=dict(color="#2563eb", width=2.0),
        hovertemplate="EMA 21: ₹%{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=plot_df.index,
        y=plot_df[f"sma_{SMA_PERIOD}"],
        mode="lines",
        name=f"SMA {SMA_PERIOD}",
        line=dict(color="#f59e0b", width=2.0),
        hovertemplate="SMA 50: ₹%{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=bullish.index,
        y=bullish["low"] - marker_padding,
        mode="markers",
        name="Bullish crossover",
        customdata=bullish[["close", f"ema_{EMA_PERIOD}", f"sma_{SMA_PERIOD}"]],
        marker=dict(
            symbol="triangle-up",
            color="#15803d",
            size=13,
            line=dict(color="white", width=1),
        ),
        hovertemplate=(
            "%{x|%d %b %Y}<br>Bullish crossover"
            "<br>Close: ₹%{customdata[0]:,.2f}"
            "<br>EMA 21: ₹%{customdata[1]:,.2f}"
            "<br>SMA 50: ₹%{customdata[2]:,.2f}<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=bearish.index,
        y=bearish["high"] + marker_padding,
        mode="markers",
        name="Bearish crossover",
        customdata=bearish[["close", f"ema_{EMA_PERIOD}", f"sma_{SMA_PERIOD}"]],
        marker=dict(
            symbol="triangle-down",
            color="#b91c1c",
            size=13,
            line=dict(color="white", width=1),
        ),
        hovertemplate=(
            "%{x|%d %b %Y}<br>Bearish crossover"
            "<br>Close: ₹%{customdata[0]:,.2f}"
            "<br>EMA 21: ₹%{customdata[1]:,.2f}"
            "<br>SMA 50: ₹%{customdata[2]:,.2f}<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=plot_df.index,
        y=plot_df[f"ema_{EMA_PERIOD}_angle"],
        mode="lines",
        name=f"EMA {EMA_PERIOD} angle",
        line=dict(color="#2563eb", width=1.6),
        hovertemplate="EMA 21 angle: %{y:.2f}°<extra></extra>",
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=plot_df.index,
        y=plot_df[f"sma_{SMA_PERIOD}_angle"],
        mode="lines",
        name=f"SMA {SMA_PERIOD} angle",
        line=dict(color="#f59e0b", width=1.6),
        hovertemplate="SMA 50 angle: %{y:.2f}°<extra></extra>",
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=plot_df.index,
        y=plot_df["volume"],
        name="Volume",
        marker_color=volume_colors,
        hovertemplate="Volume: %{y:,.0f}<extra></extra>",
    ),
    row=3,
    col=1,
)

fig.update_layout(
    title=dict(
        text="SUNPHARMA Daily Price — EMA 21 / SMA 50 Crossovers",
        x=0.01,
    ),
    height=950,
    template="plotly_white",
    hovermode="x unified",
    margin=dict(l=70, r=30, t=100, b=50),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig.update_xaxes(rangeslider_visible=False, rangebreaks=[dict(bounds=["sat", "mon"])])
fig.update_xaxes(
    title_text="Date",
    row=3,
    col=1,
)
fig.add_hline(y=0, line_width=1, line_dash="dot", line_color="#64748b", row=2, col=1)
fig.update_yaxes(title_text="Price (₹)", tickprefix="₹", row=1, col=1)
fig.update_yaxes(
    title_text="Angle (°)",
    range=[-90, 90],
    tickvals=[-60, -30, 0, 30, 60],
    row=2,
    col=1,
)
fig.update_yaxes(title_text="Volume", rangemode="tozero", row=3, col=1)
fig.show()

In [32]:
plot_df

,open,high,low,close,volume,open_interest,ema_21,sma_50,ema_21_angle,sma_50_angle,bullish_crossover,bearish_crossover,crossover
timestamp,,,,,,,,,,,,,
2024-08-01 00:00:00+05:30,"1,725.50","1,746.45","1,681.30","1,715.20",5258291,0,NaN,NaN,NaN,NaN,False,False,<NA>
2024-08-02 00:00:00+05:30,"1,700.00","1,741.80","1,683.85","1,731.65",2992406,0,NaN,NaN,NaN,NaN,False,False,<NA>
2024-08-05 00:00:00+05:30,"1,725.00","1,758.00","1,715.00","1,720.35",5165558,0,NaN,NaN,NaN,NaN,False,False,<NA>
2024-08-06 00:00:00+05:30,"1,725.00","1,734.00","1,703.95","1,707.55",1710278,0,NaN,NaN,NaN,NaN,False,False,<NA>
2024-08-07 00:00:00+05:30,"1,720.00","1,736.85","1,709.40","1,734.45",1922313,0,NaN,NaN,NaN,NaN,False,False,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-20 00:00:00+05:30,"1,942.00","1,958.80","1,935.00","1,956.40",1676034,0,"1,904.11","1,858.52",77.96,65.80,False,False,<NA>
2026-07-21 00:00:00+05:30,"1,952.10","1,966.50","1,938.20","1,961.90",1412579,0,"1,909.36","1,860.80",77.78,64.77,False,False,<NA>
2026-07-22 00:00:00+05:30,"1,938.00","1,951.80","1,925.20","1,943.00",1718430,0,"1,912.42","1,862.20",77.51,63.91,False,False,<NA>
